In [4]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoConfig, set_seed)
import torch
from torch.utils.data import DataLoader
import evaluate
metrics = evaluate.load("seqeval")


from tqdm.auto import tqdm

# Set random seeds
set_seed(42)

# Define hyperparameters (e.g., learning_rate, num_train_epochs, model_name)
learning_rate = 2e-5
num_train_epochs = 4
model_name = model_name = "bert-base-cased"
import gc



def read_iob2(path):
    sentences = []
    labels = []

    cur_words = []
    cur_labels = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # New sentence starts
            if not line:
                if cur_words:
                    sentences.append(cur_words)
                    labels.append(cur_labels)
                    cur_words = []
                    cur_labels = []
                continue

            # Skip metadata
            if line.startswith("#"):
                continue

            # Skip empty lines
            if not line:
                continue

            parts = line.split()

            # Safety check
            if len(parts) < 3:
                continue

            word = parts[1]
            label = parts[2]

            cur_words.append(word)
            cur_labels.append(label)

    # last sentence
    if cur_words:
        sentences.append(cur_words)
        labels.append(cur_labels)

    return sentences, labels

train_sentences, train_labels =read_iob2('project/en_ewt-ud-train.iob2')
dev_sentences, dev_labels= read_iob2("project/en_ewt-ud-dev.iob2")
test_sentences, test_labels= read_iob2("project/en_ewt-ud-test-masked.iob2")

unique_labels = sorted({tag for sent_tags in train_labels for tag in sent_tags})
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

train_dataset = Dataset.from_dict({
    "tokens": train_sentences,
    "ner_tags": train_labels
})
dev_dataset = Dataset.from_dict({
    "tokens": dev_sentences,
    "ner_tags": dev_labels
})

test_dataset = Dataset.from_dict({
    "tokens": test_sentences,
    "ner_tags": test_labels
})

# Load the tokenizer and model config
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Initialize the model with AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    config=config
)

def align_labels(labels, word_ids, label2id):
    aligned = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned.append(-100)
        elif word_id != prev_word_id:
            aligned.append(label2id[labels[word_id]])
        else:
            aligned.append(-100)
        prev_word_id = word_id

    return aligned

def tokenize_and_align_labels(sentences, labels, tokenizer, label2id):
    tokenized = tokenizer(
        sentences,
        is_split_into_words=True,
        truncation=True,
        padding=True
    )

    all_aligned_labels = []

    for i, sent_labels in enumerate(labels):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = align_labels(sent_labels, word_ids, label2id)
        all_aligned_labels.append(aligned)

    tokenized["labels"] = all_aligned_labels
    return tokenized

def preprocess(input):
    return tokenize_and_align_labels(
        input["tokens"],
        input["ner_tags"],
        tokenizer,
        label2id
    )

#function for evaluating model
def evaluate_model(model, dataloader, id2label, device):
    model.eval()

    true_predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            predictions = torch.argmax(outputs.logits, dim=-1)

            labels = batch["labels"]

            # Move tensors to CPU
            predictions = predictions.cpu().numpy()
            labels = labels.cpu().numpy()

            for pred, lab in zip(predictions, labels):

                current_preds = []
                current_labels = []

                for p, l in zip(pred, lab):

                    # Ignore special tokens/subwords
                    if l != -100:
                        current_preds.append(id2label[p])
                        current_labels.append(id2label[l])

                true_predictions.append(current_preds)
                true_labels.append(current_labels)

    results = metrics.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return results


tokenized_train_dataset = train_dataset.map(preprocess, batched=True)
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["tokens", "ner_tags"])
tokenized_train_dataset.set_format(type="torch")

tokenized_dev_dataset = dev_dataset.map(preprocess, batched=True)
tokenized_dev_dataset = tokenized_dev_dataset.remove_columns(["tokens", "ner_tags"])
tokenized_dev_dataset.set_format(type="torch")

tokenized_test_dataset = test_dataset.map(preprocess, batched=True)
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["tokens", "ner_tags"])
tokenized_test_dataset.set_format(type="torch")

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_dataloader = DataLoader(
    tokenized_train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

dev_dataloader = DataLoader(
    tokenized_dev_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)
# Move model to device (CPU/GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(device)
# Create optimizer (e.g. AdamW)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# import torch_directml
# device = torch_directml.device() 
# model.to(device)

# optimizer = torch.optim.SGD(
#     model.parameters(), 
#     lr=5e-5, 
#     momentum=0.9, 
#     nesterov=True, 
#     weight_decay=0.01
# )

# Implement the training loop
import torch
model.train()

for epoch in range(num_train_epochs):
    total_loss = 0
    
    pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}")
    
    for step, batch in pbar:
        #move batch to device, move every tensor(datapoints) in it to GPU/CPU
        batch = {k: v.to(device) for k, v in batch.items()}

        #forward pass
        #tuple-like object for hugging face 
        #loss calcukated automatically cuz we included 'labels' in the batch. 
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
         #backward pass
        loss.backward()  
        optimizer.step()
        #track loss
        total_loss += loss.item()
        # update progress bar with loss
        pbar.set_postfix({"loss": loss.item()})

    torch.cuda.empty_cache() # clear the backend just in case
    gc.collect()
    #print average loss
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} average training loss: {avg_train_loss:.4f}")
    # evaluating
    results = evaluate_model(
    model,
    dev_dataloader,
    id2label,
    device)
    print(f"Epoch {epoch+1} Validation Metrics:")
    print(f"Precision: {results['overall_precision']:.4f}")
    print(f"Recall:    {results['overall_recall']:.4f}")
    print(f"F1:        {results['overall_f1']:.4f}")
    print(f"Accuracy:  {results['overall_accuracy']:.4f}")

model.save_pretrained("baseline_model")
tokenizer.save_pretrained("baseline_model")

KeyboardInterrupt: 

In [4]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
def get_predictions(words, model, tokenizer, id2label, device):
    if not words:
        return []
    
    inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    predictions = torch.argmax(outputs.logits, dim=2)[0].cpu().tolist()
    word_ids = inputs.word_ids(batch_index=0)
    
    pred_labels = []
    last_word_idx = None
    for i, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != last_word_idx:
            pred_labels.append(id2label[predictions[i]])
            last_word_idx = word_idx

    return pred_labels

def write_sentence(f_out, lines, labels):
    for i, line in enumerate(lines):
        parts = line.split('\t')
        if len(parts) >= 3:
            parts[2] = labels[i]
            f_out.write("\t".join(parts) + "\n")
        else:
            f_out.write(line + "\n")

def predict_and_save(input_path, output_path, model, tokenizer, id2label, device):
    model.eval()
    
    with open(input_path, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)

    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out, \
         tqdm(total=total_lines, desc="Predicting") as pbar:
        
        current_words = []
        current_lines = []

        for line in f_in:
            pbar.update(1)
            raw_line = line.rstrip("\n\r")
            if not raw_line or raw_line.startswith("#"):
                if current_words:
                    labels = get_predictions(current_words, model, tokenizer, id2label, device)
                    write_sentence(f_out, current_lines, labels)
                    current_words, current_lines = [], []
                f_out.write(line)
                continue

            parts = raw_line.split('\t')
            if len(parts) >= 2:
                current_words.append(parts[1])
                current_lines.append(raw_line)

        if current_words:
            labels = get_predictions(current_words, model, tokenizer, id2label, device)
            write_sentence(f_out, current_lines, labels)

predict_and_save(
    'project/en_ewt-ud-dev.iob2',
    'project/en_ewt-ud-dev-predictions.iob2',
    model, tokenizer, id2label, device
)
 
def evaluate_model(model, dataloader, id2label, device):
    model.eval()

    true_predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)

            predictions = torch.argmax(outputs.logits, dim=-1)

            labels = batch["labels"]

            # Move tensors to CPU
            predictions = predictions.cpu().numpy()
            labels = labels.cpu().numpy()

            for pred, lab in zip(predictions, labels):

                current_preds = []
                current_labels = []

                for p, l in zip(pred, lab):

                    # Ignore special tokens/subwords
                    if l != -100:
                        current_preds.append(id2label[p])
                        current_labels.append(id2label[l])

                true_predictions.append(current_preds)
                true_labels.append(current_labels)

    results = metrics.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return results

Predicting: 100%|██████████| 31470/31470 [00:10<00:00, 2874.29it/s]


In [ ]:
from datasets import concatenate_datasets
fiction_sentences, fiction_labels= read_iob2("fiction_dataset.iob2")

#Some labels were wrong, bugfixing
valid_labels = {
    "O",
    "B-ARTIFACT", "I-ARTIFACT",
    "B-CREATURE", "I-CREATURE",
    "B-SPELL", "I-SPELL",
    "B-LOC", "I-LOC",
    "B-ORG", "I-ORG",
    "B-PER", "I-PER"
}

fixed = 0

for sent_labels in fiction_labels:
    for i, label in enumerate(sent_labels):

        # fix lowercase o
        if label == "o":
            sent_labels[i] = "O"
            fixed += 1

        # replace unknown labels
        elif sent_labels[i] not in valid_labels:
            print(f"Replacing bad label: {sent_labels[i]} -> O")
            sent_labels[i] = "O"
            fixed += 1

print(f"Fixed {fixed} labels")

fiction_dataset = Dataset.from_dict({"tokens": fiction_sentences,
                                     "ner_tags": fiction_labels})

split_dataset= fiction_dataset.train_test_split(test_size=0.2, seed=42)
train_fiction= split_dataset['train']
test_fiction= split_dataset["test"]

combined_train_dataset= concatenate_datasets([train_dataset, train_fiction])

unique_labels = sorted({
    tag
    for sent_tags in combined_train_dataset["ner_tags"]
    for tag in sent_tags
})

label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
# Load the tokenizer and model config
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Initialize the model with AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    config=config
)
tokenized_train_dataset = combined_train_dataset.map(preprocess, batched=True)
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["tokens", "ner_tags"])
tokenized_train_dataset.set_format(type="torch")


tokenized_test_dataset = test_fiction.map(preprocess, batched=True)
tokenized_test_dataset = tokenized_test_dataset.remove_columns(["tokens", "ner_tags"])
tokenized_test_dataset.set_format(type="torch")

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_dataloader = DataLoader(
    tokenized_train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    tokenized_test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)
# Move model to device (CPU/GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(device)
# Create optimizer (e.g. AdamW)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()

for epoch in range(num_train_epochs):
    total_loss = 0
    
    pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}")
    
    for step, batch in pbar:
        #move batch to device, move every tensor(datapoints) in it to GPU/CPU
        batch = {k: v.to(device) for k, v in batch.items()}

        #forward pass
        #tuple-like object for hugging face 
        #loss calcukated automatically cuz we included 'labels' in the batch. 
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
         #backward pass 
        loss.backward()  
        optimizer.step()
        #track loss
        total_loss += loss.item()
        # update progress bar with loss
        pbar.set_postfix({"loss": loss.item()})

    torch.cuda.empty_cache() # clear the backend just in case
    gc.collect()
    #print average loss
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} average training loss: {avg_train_loss:.4f}")
    results = evaluate_model(
    model,
    dev_dataloader,
    id2label,
    device)

    print(f"Epoch {epoch+1} Validation Metrics:")
    print(f"Precision: {results['overall_precision']:.4f}")
    print(f"Recall:    {results['overall_recall']:.4f}")
    print(f"F1:        {results['overall_f1']:.4f}")
    print(f"Accuracy:  {results['overall_accuracy']:.4f}")

model.save_pretrained("fantasy_adapted_model")
tokenizer.save_pretrained("fantasy_adapted_model")

Replacing bad label: blot -> O
Replacing bad label: blot -> O
Replacing bad label: Malfoys -> O
Fixed 6 labels


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12206.07it/s]
BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

cuda


Epoch 1: 100%|██████████| 799/799 [01:31<00:00,  8.70it/s, loss=0.0142] 


Epoch 1 average training loss: 0.1049


...

Epoch 2 average training loss: 0.0286


Epoch 3: 100%|██████████| 799/799 [01:30<00:00,  8.84it/s, loss=0.0104]  


Epoch 3 average training loss: 0.0144


Epoch 4: 100%|██████████| 799/799 [01:30<00:00,  8.81it/s, loss=0.00205] 


Epoch 4 average training loss: 0.0084


In [29]:
def save_dataset_as_iob2(dataset, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for tokens, labels in zip(dataset["tokens"], dataset["ner_tags"]):
            for i, (token, label) in enumerate(zip(tokens, labels), start=1):
                f.write(f"{i}\t{token}\t{label}\n")
            f.write("\n")

save_dataset_as_iob2(test_fiction, "fiction_test.iob2")

predict_and_save(
    "fiction_test.iob2",
    "fiction_test_predictions.iob2",
    model,
    tokenizer,
    id2label,
    device
)

Predicting: 100%|██████████| 1207/1207 [00:00<00:00, 1962.11it/s]
